### 🖼️ Dataset Thumbnails (Major-TOM___Core-S2L2A)

![Thumbnail](../thumbnails/Major-TOM___Core-S2L2A_01.png)

In [ ]:
import numpy as np
import pandas as pd
import random
from datasets import load_dataset, get_dataset_config_names
import matplotlib.pyplot as plt
from PIL import Image

# ==================================================================================
# 🌐 데이터셋 분석 프로젝트: 우주 사진 탐험가 되기 🛰️
# ==================================================================================
# [데이터셋명] Major-TOM/Core-S2L2A
# [주제] Sentinel-2 (센티넬-2) 위성 사진 데이터 (Level 2A)
# [설명] 이 데이터셋은 지표면을 촬영한 고해상도(10m) 다중 분광 이미지 패치(patch)들을 담고 있어요.
#         지구 관측, 환경 모니터링, 농업 분석 등 거대한 AI 프로젝트의 기반이 되는 소중한 자료랍니다.
#         초보자 관점에서는 '어떤 사진이 가장 깨끗한지', '이 지역의 식생은 어떤지'를 탐색하는 것부터 시작할 수 있어요!
# ==================================================================================

# === 설정 변수 ===
DATASET_NAME = "Major-TOM/Core-S2L2A"
SAMPLE_COUNT = 5  # 전체 데이터셋 중 재미로 분석해 볼 샘플 개수
# ====================


def load_dataset_safely(dataset_id, split_name, streaming_attempt):
    """
    스트리밍 모드와 일반 모드를 오가며 데이터셋을 안전하게 로드하는 함수입니다.
    (🌟 초보자도 데이터 로딩에 실패하지 않도록 도와줄 안전장치!)
    """
    print("\n✨ 데이터셋 로딩을 시도합니다...")
    
    try:
        # 1. 🌟 최우선 시도: 스트리밍 모드 (가장 빠르고 메모리 효율적!)
        print("   -> 💨 스트리밍 모드 (streaming=True)로 로드 시도...")
        dataset = load_dataset(dataset_id, split=split_name, streaming=True)
        return dataset
    except Exception as e:
        # 2. 🚫 실패 시: 일반 모드 (스트리밍이 안 되면, 몇 개만 다운로드 받아서라도 진행!)
        print(f"   ❌ 스트리밍 모드 실패 ({e}). 일반 모드로 {SAMPLE_COUNT}개만 로드합니다.")
        try:
            dataset = load_dataset(dataset_id, split=split_name, streaming=False)
            # 전체 데이터셋을 메모리에 올릴 수 없으므로, 아예 일부만 샘플링 합니다.
            return dataset.select(range(min(SAMPLE_COUNT + 1, len(dataset))))
        except Exception as e_fallback:
            print(f"   🚨 치명적인 오류가 발생했습니다: {e_fallback}")
            return None


# 1. 데이터셋 로드 및 샘플링
# ----------------------------------------------------------
# 우리는 전체 데이터를 다 볼 필요가 없어요. 흥미로운 5개의 샘플만 골라 분석해 볼게요!
dataset = load_dataset_safely(DATASET_NAME, 'train', 'train')

if dataset is None:
    print("😭 데이터셋을 로드할 수 없어 실습을 진행할 수 없습니다. 잠시 후 다시 시도해주세요.")
    exit()

# 2. 스트리밍/비스트리밍 데이터셋에 맞는 샘플 추출 방식 적용
# ----------------------------------------------------------
print("🧩 데이터셋 준비: 실제로 분석할 샘플들을 뽑아봅시다.")

if hasattr(dataset, "take"):
    # .take()가 존재하면 -> 스트리밍 데이터셋 (IterableDataset)
    print("   (ℹ️ 스트리밍 데이터셋입니다. next()를 사용해 하나씩 가져옵니다.)")
    sample_iterator = dataset.take(SAMPLE_COUNT)
    sample_list = list(sample_iterator) # 🌟 리스트로 변환하여 반복문에서 사용
else:
    # 일반 데이터셋 (Dataset) -> .select()로 이미 샘플링되어 있을 가능성이 높음
    print("   (ℹ️ 일반 데이터셋입니다. 이미 충분히 샘플링되어 있습니다.)")
    # load_dataset_safely에서 이미 샘플링되었으므로, 리스트로 변환합니다.
    sample_list = list(dataset)

if not sample_list:
    print("⚠️ 분석할 샘플이 없습니다. 데이터셋 로딩을 다시 확인해주세요.")
    exit()

# 3. 데이터 분석 (AI 실습 핵심 구간)
# ----------------------------------------------------------

def analyze_satellite_patch(sample_data: dict, index: int):
    """
    하나의 위성 사진 패치를 받아서 재미있는 'AI 판별'을 수행하는 함수입니다.
    (🌟 이 함수가 바로 AI가 데이터를 해석하는 과정과 같습니다!)
    """
    print("\n" + "="*80)
    print(f"🔬 [분석 샘플 {index + 1}/{len(sample_list)}] - 분석 시작!")

    # A. 클라우드 마스크 분석 (가장 중요!)
    cloud_mask = sample_data.get('cloud_mask')
    if cloud_mask is not None:
        # binary 데이터가 numpy 배열일 수 있으므로, 평균을 내어 전체적인 구름 비율을 추정합니다.
        # 값이 클수록 구름이 많다고 가정합니다. (여기서는 0 또는 1 값으로만 가정하고 간단히 체크)
        is_cloudy = np.mean(cloud_mask) > 0.5
        print(f"☁️ 날씨 판정 (Cloud Mask): {'🚨 구름이 많아 데이터 해석에 주의가 필요해요!' if is_cloudy else '✅ 맑은 날, 데이터 품질이 좋아 보여요!'}")
    else:
        print("☁️ 날씨 판정: 클라우드 마스크 정보가 없어 판별하기 어려워요.")

    # B. 식생 지수 분석 시뮬레이션 (NDVI 개념 차용)
    # 초보자에게 가장 재미있는 분석 영역입니다.
    try:
        # 대표적인 식생 지표 밴드 (B03: Green, B08: Near-Infrared)를 가져옵니다.
        b03 = sample_data['B03']
        b08 = sample_data['B08']

        # ⚠️ 참고: 실제 환경에서는 값에 대한 수학적 계산(나눗셈 등)이 필요하지만,
        # 여기서는 데이터가 로드되었다는 사실 자체를 '정보의 풍부함'으로 판단합니다.
        
        # 가상의 건강 점수 계산 (단순 합산으로 구현)
        health_score = np.sum(b03) + np.sum(b08)
        
        print(f"🌿 생명력 점수 (B03 + B08): {health_score:.2f} (값이 높을수록 활발한 식생을 나타낼 수 있어요)")
        if health_score > 0.5:
            print("   -> 💚 분석가 의견: 식생 활동이 활발한 지역으로 추정됩니다! (농작지 또는 녹지)")
        else:
            print("   -> 💛 분석가 의견: 식생 활동이 적거나 건조한 지역일 수 있습니다. 다른 요인(토양, 수분)을 함께 고려해야 해요.")

    except KeyError as e:
        print(f"⚠️ 분석 경고: 필수 밴드 {e}를 찾을 수 없어 '생명력 점수' 계산을 건너뜁니다.")
    except Exception as e:
        print(f"⚠️ 분석 경고: 계산 중 오류가 발생했습니다. ({e})")


    # C. 시각화 (가장 직관적인 탐색)
    print("\n🖼️ 시각화: 샘플의 모습을 한번 살펴볼까요?")
    try:
        img_thumb = sample_data.get('thumbnail')
        if isinstance(img_thumb, Image.Image):
            plt.figure(figsize=(8, 8))
            plt.imshow(np.array(img_thumb))
            plt.title(f"Sample Image Thumbnail (ID: {sample_data.get('product_id')[:5]}...)")
            plt.axis('off')
            plt.show()
        else:
            print("🖼️ 썸네일 이미지를 찾을 수 없거나 표시할 수 없습니다.")

    except Exception as e:
        print(f"🖼️ 시각화 중 오류 발생: {e}")


# 4. 반복 분석 실행
# ----------------------------------------------------------
print("\n" + "="*80)
print("🚀 스크립트 실행: 우주 사진 분석가 가동!")
print("="*80)

for i, sample in enumerate(sample_list):
    analyze_satellite_patch(sample, i)

print("\n✨ 모든 샘플 분석을 마쳤습니다!")
print("✨ 축하드립니다! 이제 여러분은 위성 데이터의 기본적인 '탐색자'가 되셨어요. 다음 단계에서는 이 데이터를 기반으로 AI 모델을 훈련시켜볼 수 있답니다!")